In [79]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import subprocess
import sys
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ParameterSampler
import gc
from sklearn.base import clone

# Install pgeocode for geographic distance calculation
try:
    import pgeocode
except ImportError:
    print("Installing pgeocode...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pgeocode"], check=True)
    import pgeocode

try:
    from ethnicolr import census_ln
except ImportError:
    print("Installing ethnicolr...")
    # Note: ethnicolr requires TensorFlow. This installation might take a moment.
    subprocess.run([sys.executable, "-m", "pip", "install", "ethnicolr"], check=True)
    from ethnicolr import census_ln

import warnings
#warnings.filterwarnings("ignore", category=FutureWarning)

import os
from sklearn.model_selection import ParameterGrid, cross_validate
from sklearn.model_selection import ParameterSampler
import joblib
import traceback

try:
    import lightgbm as lgb
except ImportError:
    print("Installing lightgbm...")
    subprocess.run([sys.executable, "-m", "pip", "install", "lightgbm"], check=True)
    import lightgbm as lgb

try:
    import xgboost as xgb
except ImportError:
    print("Installing xgboost...")
    subprocess.run([sys.executable, "-m", "pip", "install", "xgboost"], check=True)
    import xgboost as xgb

try:
    import catboost as cb
except ImportError:
    print("Installing catboost...")
    subprocess.run([sys.executable, "-m", "pip", "install", "catboost"], check=True)
    import catboost as cb


In [80]:
#file locations
parquet_file_paths={
    "patient": r"Client_Data_files\Parquets\synthetic_patients.parquet",
    "encounter": r"Client_Data_files\Parquets\synthetic_encounters.parquet",
    "hospitals": r"Client_Data_files\Parquets\synthetic_hospitals.parquet",
    "provider": r"Client_Data_files\Parquets\synthetic_providers.parquet",    
}

# Reading the parquet files
patient_df = pd.read_parquet(parquet_file_paths['patient'])
encounter_df = pd.read_parquet(parquet_file_paths['encounter'])
hospital_df = pd.read_parquet(parquet_file_paths['hospitals'])
provider_df = pd.read_parquet(parquet_file_paths['provider'])

print("Dataframes loaded successfully.")
print(f"Patient DF shape: {patient_df.shape}")
print(f"Encounter DF shape: {encounter_df.shape}")
print(f"Provider DF shape: {provider_df.shape}")
print(f"Hospital DF shape: {hospital_df.shape}")

Dataframes loaded successfully.
Patient DF shape: (100000, 16)
Encounter DF shape: (200000, 17)
Provider DF shape: (5000, 19)
Hospital DF shape: (200, 14)


In [81]:
# Creating a data map for easy access
data_map={
    "patient": patient_df,
    "encounter": encounter_df,
    "hospitals": hospital_df,
    "provider": provider_df
}

def describe_Cols(data_map):
    for key, df in data_map.items():
        print(f"Dataframe: {key}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")    
        for col in df.columns:
            if df[col].isna().sum() > 0:
                print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
        print("")

In [82]:
describe_Cols(data_map)

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

In [83]:
race_mapping={
    'white': 'White',
    'black': 'Black or African American',
    'api': 'Asian',    
    'aian': 'Native American',
    '2prace': 'Other'
}

In [84]:
patient_df_temp=pd.DataFrame()
patient_df_temp=patient_df[patient_df['race']=='Hispanic or Latino'][['patient_id','first_name','last_name','race','ethnicity']].copy()
patient_race_pred=census_ln(patient_df_temp, 'last_name')

race_cols=['pctwhite','pctblack','pctapi','pctaian','pct2prace']
patient_race_pred['derived_race'] = patient_race_pred[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)

# Create a mapping from patient_id to derived_race
id_to_derived_race = dict(zip(patient_race_pred['patient_id'], patient_race_pred['derived_race']))

# Update the race column only for Hispanic or Latino patients
patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'race'] = \
    patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'patient_id'].map(id_to_derived_race)

2025-10-10 20:58:35,584 - INFO - Preserving 12965 duplicate rows based on column 'last_name'
2025-10-10 20:58:35,584 - INFO - Data filtering summary: 12997 → 12997 rows (kept 100.0%)
2025-10-10 20:58:35,601 - INFO - Merging demographic data for 12997 records...
2025-10-10 20:58:35,676 - INFO - Matched 12997 of 12997 rows (100.0%)
2025-10-10 20:58:35,676 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


In [85]:
provider_race_predictions = census_ln(provider_df, 'last_name')

# Derive race for the provider_df as it is missing from the source data
print("Deriving race for providers from last names...")
race_cols = ['pctwhite','pctblack','pctapi','pctaian','pct2prace']
provider_df['provider_race'] = provider_race_predictions[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
print("Provider race derivation complete.")

# Deriving provider ethnicity from the race_predictions
print("Deriving ethnicity for providers from race predictions...")
provider_df['provider_ethnicity'] = provider_race_predictions['pcthispanic'].apply(lambda x: 'Hispanic or Latino' if float(x) >= 50 else 'Not Hispanic or Latino')
print("Provider ethnicity derivation complete.")


2025-10-10 20:58:35,744 - INFO - Preserving 4968 duplicate rows based on column 'last_name'
2025-10-10 20:58:35,746 - INFO - Data filtering summary: 5000 → 5000 rows (kept 100.0%)
2025-10-10 20:58:35,746 - INFO - Merging demographic data for 5000 records...
2025-10-10 20:58:35,822 - INFO - Matched 5000 of 5000 rows (100.0%)
2025-10-10 20:58:35,824 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


Deriving race for providers from last names...
Provider race derivation complete.
Deriving ethnicity for providers from race predictions...
Provider ethnicity derivation complete.


In [86]:
describe_Cols(data_map)

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

### Step 2: Feature Engineering

Here, we merge the datasets and create the features our model will learn from. This includes cultural matches, language matches, and geographic distance.

In [87]:
# Merge all data into a single master DataFrame for training
master_df = pd.merge(encounter_df, patient_df, on='patient_id',suffixes=('', '_pat'))
master_df = pd.merge(master_df, provider_df, on='provider_id',suffixes=('', '_prov'))
master_df = pd.merge(master_df, hospital_df, left_on='hospital_affiliation', right_on='hospital_id',suffixes=('', '_hosp'))

print("Master DataFrame created with shape:", master_df.shape)

Master DataFrame created with shape: (200000, 66)


In [88]:
df_order_list=['encounter', 'patient', 'provider', 'hospitals',]
prev=0
curr=0
for  i, df_name in enumerate(df_order_list, start=0):
    curr=curr+data_map[df_name].shape[1]-([0,1,1,4][i] if i < len([0,1,3,4]) else 0)
    print(f"{df_name}: {master_df.columns[prev:curr].tolist()}")
    print()
    prev=curr

encounter: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

patient: ['first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language_pat', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background_pat', 'preferred_provider_language', 'cultural_preferences']

provider: ['npi_number', 'first_name_prov', 'last_name_prov', 'specialty', 'practice_zip_code', 'years_experience', 'medical_school_country', 'board_certified', 'languages_spoken_prov', 'interpreter_services', 'cultural_certifications', 'minority_health_experience', 'community_involvement', 'patient_satisfaction_score', 'communication_rating', 'cultural_competency_rating_prov'

In [89]:

# --- Engineer the Stateless Features ---

master_df['encounter_date'] = pd.to_datetime(master_df['encounter_date'])

# Stateless Cultural Features
master_df['race_match'] = (master_df['race'] == master_df['provider_race']).astype(int)
master_df['ethnicity_match'] = (master_df['ethnicity'] == master_df['provider_ethnicity']).astype(int)
master_df['language_match'] = (master_df['language_match'] == True).astype(int)



# Geographic Feature
dist = pgeocode.GeoDistance('US') # Assuming US zip codes
# Calculate distance between patient and provider zip codes
master_df['distance_km'] = dist.query_postal_code(
    master_df['zip_code'].astype(str).tolist(), 
    master_df['zip_code_hosp'].astype(str).tolist()
)
# Calculate the mean distance for each provider specialty
# The .transform('mean') creates a Series with the same index as master_df,
mean_dist_by_specialty = master_df.groupby('specialty')['distance_km'].transform('mean')

# Now, fill the missing distances using these specialty-specific averages
master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)

# If any specialties had NO valid distances, there might still be NaNs.
# Fill any remaining with the overall mean as a final fallback.
master_df['distance_km'].fillna(master_df['distance_km'].mean(), inplace=True)


C:\Users\jerry\AppData\Local\Temp\ipykernel_31236\4275704363.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)
C:\Users\jerry\AppData\Local\Temp\ipykernel_31236\4275704363.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [90]:
# Create a function to apply the transformations consistently
def create_stateful_features(df,  min_dist, max_dist):
    df_eng = df.copy() # Work on a copy to avoid SettingWithCopyWarning
    
    
    # Proximity Score
    df_eng['proximity_score'] = 1 - ((df_eng['distance_km'] - min_dist) / (max_dist - min_dist))
    df_eng['proximity_score'] = df_eng['proximity_score'].clip(0, 1)
    
    return df_eng


### Step 3: Model Configuration Block
**This is the main section to edit.** Add, remove, or modify the models and their hyperparameter grids in this dictionary.

In [91]:
model_configs = {
    'RandomForest': {
        'estimator': RandomForestRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [50, 100, 150, 200],
            'max_depth': [10, 20, 30, None],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 1.0]
        },
        'code': 'RFR'
    },
    'LightGBM': {
        'estimator': lgb.LGBMRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [20, 31, 40],
            'max_depth': [-1, 10, 20]
        },
        'code': 'LGBM'
    },
    'XGBoost': {
        'estimator': xgb.XGBRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8],
            'colsample_bytree': [0.7, 0.8]
        },
        'code': 'XGB'
    },
    'CatBoost': {
        'estimator': cb.CatBoostRegressor(random_state=42, thread_count=-1, verbose=0),
        'param_grid': {
            'iterations': [100, 200, 300], 
            'learning_rate': [0.01, 0.05, 0.1],
            'depth': [4, 6, 8],
            'l2_leaf_reg': [1, 3, 5, 7]
            },
        'code': 'CAT',
    }

}

print(f"Prepared configurations for {list(model_configs.keys())}")

Prepared configurations for ['RandomForest', 'LightGBM', 'XGBoost', 'CatBoost']


In [92]:
# Define the features to be used by the models
features_stateless = [
    'years_experience',
    'cultural_competency_rating_prov',
    'communication_rating',
    'race_match',
    'ethnicity_match',
    'language_match',    
    'interpreter_services_24_7',
    'patient_satisfaction_score',
    'age',
    'readmission_rate',
    'minority_health_experience',
        
]

features_stateful = [    
    'proximity_score',  
]

features=features_stateless + features_stateful


# --- Engineer the Target Variable ---

scaler = MinMaxScaler()
master_df[['satisfaction_norm', 'adherence_norm']] = scaler.fit_transform(
    master_df[['patient_satisfaction', 'treatment_adherence']]
)

# 2. Define weights and create the composite score
adherence_weight = 0.5
satisfaction_weight = 0.5
master_df['success_score'] = (
    master_df['adherence_norm'] * adherence_weight +
    master_df['satisfaction_norm'] * satisfaction_weight
)

target = 'success_score'




In [93]:
# Get the unique preference categories to loop through
unique_preferences = master_df['cultural_preferences'].unique()
print(f"Unique cultural preferences found:({len(unique_preferences)}) {unique_preferences.tolist()}")

Unique cultural preferences found:(4) ['No Specific Preference', 'Culturally Similar Provider', 'Culturally Similar Provider; Same Language Provider', 'Same Language Provider']


### Step 3: Training and Testing the Model

This is the core machine learning section. We split our data, train the model, and then test it on unseen data to validate its performance. The feature importances are the **learned weights**.

In [94]:
# --- Dictionaries to Store All Results ---
all_results = {}

models_to_runs = [
    'RandomForest', 
    'LightGBM', 
    'XGBoost',
    'CatBoost'
    ]

for model_name, config in model_configs.items():
    if model_name not in models_to_runs:
        continue

    print(f"\n{'='*20} RUNNING EXPERIMENT FOR MODEL: {model_name.upper()} {'='*20}")
    all_results[model_name] = {
        'trained_models': {},
        'learned_weights_dict': {},
        'best_hyperparameters': {},
        'test_metrics_dict': {},
        'feature_engineering_params': {}, # To store scaling values like min/max dist
        'all_run_logs_dict': {}
    }

    for i, preference in enumerate(unique_preferences):
        print(f"Training model for cultural preference: {preference} ({i+1}/{len(unique_preferences)})--------------------------------------")
        # Filter the master_df for the current cultural preference
        segment_df = master_df[master_df['cultural_preferences'] == preference].copy()

        # Check if the segment is large enough to train a model
        if len(segment_df) < 100: # You can adjust this threshold
            print(f"Segment is too small to train a reliable model. Skipping.\n")
            continue

        # Split the data into training and testing sets
        X_segment = segment_df.drop(columns=[target])
        y_segment = segment_df[target]

        # 1. Split data into a training+validation set (80%) and a final test set (20%)
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X_segment, y_segment, test_size=0.2, random_state=42
        )

        # Learn parameters ONLY from the training set
        print("Learning feature engineering parameters from the training set...")
        # avg_adherence_train = X_train_val['treatment_adherence'].mean()
        min_dist_train = X_train_val['distance_km'].min()
        max_dist_train = X_train_val['distance_km'].max()

        # Store these learned parameters for later use in production/inference
        all_results[model_name]['feature_engineering_params'][preference] = {
            # 'avg_adherence': avg_adherence_train,
            'min_dist': min_dist_train,
            'max_dist': max_dist_train
        }

        # Apply the function to both train and test sets
        X_train_val = create_stateful_features(X_train_val,  min_dist_train, max_dist_train)
        X_test = create_stateful_features(X_test,  min_dist_train, max_dist_train)

        
        X_train_val = create_stateful_features(X_train_val,  min_dist_train, max_dist_train)
        X_test = create_stateful_features(X_test,  min_dist_train, max_dist_train)

        X_train_val=X_train_val[features]
        X_test=X_test[features]

        print(f"training on features: {X_train_val.columns.tolist()}")
        print(f"Training+Validation set size: {X_train_val.shape[0]} samples")
        print(f"Test set size: {X_test.shape[0]} samples")


        # Initialize the model
        base_model = config['estimator']
        model = clone(base_model)
        param_grid = config['param_grid']
        model_code = config['code']

        n_iterations = 50 
        param_sampler = ParameterSampler(
            param_grid, 
            n_iter=n_iterations, 
            random_state=42)
        
        total_combinations = n_iterations

        all_run_logs = []
        for run_idx, params in enumerate(param_sampler):
            print(f"  > Running trial {run_idx + 1}/{total_combinations}...", end='\r')
            model.set_params(**params)

            # Define the metrics to calculate during cross-validation
            scoring_metrics = {
                'neg_mse': 'neg_mean_squared_error',
                'mae': 'neg_mean_absolute_error',
                'r2': 'r2'
            }

            cv_results = cross_validate(
                model, 
                X_train_val, y_train_val,
                scoring=scoring_metrics,
                cv=5,
                return_estimator=True
            )

            # Calculate mean feature importances across the 5 folds
            fold_importances = [est.feature_importances_ for est in cv_results['estimator']]
            mean_importances = np.mean(fold_importances, axis=0)

             # Store all the results in our log list
            all_run_logs.append({
                'params': params,
                'mean_rmse': np.sqrt(-np.mean(cv_results['test_neg_mse'])),
                'mean_mae': -np.mean(cv_results['test_mae']),
                'mean_r2': np.mean(cv_results['test_r2']),
                'feature_importances': dict(zip(features, mean_importances))
            })

        print(f"\nSearch complete after {total_combinations} trials. Analyzing results...")

        # 5. Convert logs to a DataFrame for easy analysis
        results_df = pd.DataFrame(all_run_logs)
        results_df = results_df.sort_values(by='mean_rmse', ascending=True)

        # 6. Select the best model based on RMSE
        best_run = results_df.iloc[0]
        best_params = best_run['params']
        print(f"Best parameters found: {best_params}")

        all_results[model_name]['best_hyperparameters'][preference] = best_params

        best_model = model.set_params(**best_params)
        best_model.fit(X_train_val, y_train_val)
        all_results[model_name]['trained_models'][preference] = best_model
        all_results[model_name]['all_run_logs_dict'][preference] = results_df

        learned_weights = best_run['feature_importances']
        all_results[model_name]['learned_weights_dict'][preference] = learned_weights
    
        print("\n--- Learned Feature Weights for this Segment ---")
        print(learned_weights)

        # --- Final Testing Section ---

        print("\n--- Final Evaluation on the Held-Out Test Set ---")
        print(f"Test set size: {X_test.shape[0]} encounters")
        
        # 6. Make predictions on the unseen test data
        final_predictions = best_model.predict(X_test)

        # 7. Evaluate the final model's performance
        rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
        mae = mean_absolute_error(y_test, final_predictions)
        r2 = r2_score(y_test, final_predictions)

        print("\n--- Final Model Validation Metrics ---")
        print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
        print(f"Mean Absolute Error (MAE):     {mae:.4f}")
        print(f"R-squared (R²):                {r2:.4f}")

        # store the test run results 
        # You can store these metrics in a dictionary or DataFrame if needed
        test_metrics = {
            'preference': preference,
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'test_set_size': X_test.shape[0]
        }

        all_results[model_name]['test_metrics_dict'][preference] = test_metrics

        print(f"\n--- Best model for '{preference}' trained and stored ----------------------------------------------------------------")

        # Clean up to free memory
        del segment_df, X_segment, y_segment
        del X_train_val, X_test, y_train_val, y_test
        del best_model, model, base_model
        gc.collect()
        print("Memory cleaned up after this preference.----------------------------------------------------\n")


==================== RUNNING EXPERIMENT FOR MODEL: RANDOMFOREST ====================
Training model for cultural preference: No Specific Preference (1/4)--------------------------------------
Learning feature engineering parameters from the training set...
training on features: ['years_experience', 'cultural_competency_rating_prov', 'communication_rating', 'race_match', 'ethnicity_match', 'language_match', 'interpreter_services_24_7', 'patient_satisfaction_score', 'age', 'readmission_rate', 'minority_health_experience', 'proximity_score']
Training+Validation set size: 69151 samples
Test set size: 17288 samples
  > Running trial 50/50...
Search complete after 50 trials. Analyzing results...
Best parameters found: {'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 10}

--- Learned Feature Weights for this Segment ---
{'years_experience': np.float64(0.09052048950119436), 'cultural_competency_rating_prov': np.float64(0.18481663067367654), 'communication_rati

In [ ]:
all_results.keys()

dict_keys(['RandomForest', 'LightGBM', 'XGBoost'])

In [ ]:
all_results['LightGBM'].keys()

dict_keys(['trained_models', 'learned_weights_dict', 'best_hyperparameters', 'test_metrics_dict', 'feature_engineering_params', 'all_run_logs_dict'])

In [ ]:
pd.DataFrame(all_results['CatBoost']['test_metrics_dict'])

,No Specific Preference,Culturally Similar Provider,Culturally Similar Provider; Same Language Provider,Same Language Provider
preference,No Specific Preference,Culturally Similar Provider,Culturally Similar Provider; Same Language Pro...,Same Language Provider
rmse,0.109278,0.108972,0.13961,0.139558
mae,0.090489,0.09004,0.114665,0.113746
r2,0.003162,0.001534,0.373461,0.373738
test_set_size,17288,13852,3916,4945


In [ ]:
test_metrics_df = pd.DataFrame(all_results['LightGBM']['test_metrics_dict'])

test_metrics_df

,No Specific Preference,Culturally Similar Provider,Culturally Similar Provider; Same Language Provider,Same Language Provider
preference,No Specific Preference,Culturally Similar Provider,Culturally Similar Provider; Same Language Pro...,Same Language Provider
rmse,0.109322,0.109009,0.13996,0.139676
mae,0.090526,0.090073,0.114981,0.113916
r2,0.002375,0.000853,0.370316,0.372676
test_set_size,17288,13852,3916,4945


In [95]:
from openpyxl.styles import Font # Import Font for styling
from datetime import datetime 

# Save each model's results into separate sheets in the same Excel file

# --- Report Generation Section ---
# This code should be run AFTER your training loop is complete.

print("\\n--- Generating Final Model Summary Report ---")

# Define report name
report_name = "Composite_score"

file_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
report_filename = f"{report_name}_{file_suffix}.xlsx"

try:
    # create a folder 'Summary_reports' if it doesn't exist
    if not os.path.exists('Summary_reports'):
        os.makedirs('Summary_reports')
    
    report_path = os.path.join('Summary_reports', report_filename)
    with pd.ExcelWriter(report_path, engine='openpyxl', mode='a' if os.path.exists(report_path) else 'w') as writer:
        for model_name, results in all_results.items():
            print(f"Saving report for model: {model_name}")
            model_code = model_configs[model_name]['code']            
            sheet_name = model_name

            workbook = writer.book
            # Create the sheet if it doesn't exist, otherwise get it
            if sheet_name not in workbook.sheetnames:
                worksheet = workbook.create_sheet(sheet_name)
            else:
                worksheet = workbook[sheet_name]
            writer.sheets[sheet_name] = worksheet
            
            # a) Add Model Name (Title)        
            worksheet['A1'] = model_name
            worksheet['A1'].font = Font(bold=True, size=14)
            # b) Add Model Code
            worksheet['A2'] = model_code
            worksheet['A2'].font = Font(bold=True, size=12)
            # c) Add Date and Time of Report Generation
            worksheet['A3'] = f"Report generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

            # add model summary
            #
            #
            #

            # Write each dataframe from the all_results[model_name] dictionary in the same sheet
            current_row = 11
            table_spacing = 3  # Rows between tables

            # Write test_metrics_dict
            test_metrics_df = pd.DataFrame(all_results[model_name]['test_metrics_dict']).reset_index().rename(columns={'index': 'Metric'})            
            test_metrics_df.to_excel(writer, sheet_name=sheet_name, startrow=current_row, index=True)

            current_row += test_metrics_df.shape[0] + table_spacing  # Move down for the next table

            # Write learned_weights_dict
            weights_df = pd.DataFrame(all_results[model_name]['learned_weights_dict']).reset_index().rename(columns={'index': 'Feature'})
            weights_df.to_excel(writer, sheet_name=sheet_name, startrow=current_row, index=True)

            current_row += weights_df.shape[0] + table_spacing  # Move down for the next table

            # Write best_hyperparameters
            hyperparams_df = pd.DataFrame(all_results[model_name]['best_hyperparameters']).reset_index().rename(columns={'index': 'hyperparameter'})
            hyperparams_df.to_excel(writer, sheet_name=sheet_name, startrow=current_row, index=True)

            current_row += hyperparams_df.shape[0] + table_spacing  # Move down for the next table

            sheet_name_logs = f"{model_name}_Logs"
            if sheet_name_logs not in workbook.sheetnames:
                worksheet_logs = workbook.create_sheet(sheet_name_logs)
            else:
                worksheet_logs = workbook[sheet_name_logs]
            writer.sheets[sheet_name_logs] = worksheet_logs
            current_row = 3  # Reset to the top of the logs sheet

            # Write all run logs for each preference
            for pref, logs_df in all_results[model_name]['all_run_logs_dict'].items():
                logs_df.to_excel(writer, sheet_name=sheet_name_logs, startrow=current_row, index=False)
                current_row += logs_df.shape[0] + table_spacing  # Move down for the next table
            
            print(f"Report for model {model_name} saved successfully.")

    print(f"All reports generated and saved to {report_path}")

            
except Exception as e:
    #print the error traceback
    
    traceback.print_exc()
    print(f"Error saving report: {e}")

    print(f"Report generated and saved to {report_path}")
        

\n--- Generating Final Model Summary Report ---
Saving report for model: RandomForest
Report for model RandomForest saved successfully.
Saving report for model: LightGBM
Report for model LightGBM saved successfully.
Saving report for model: XGBoost
Report for model XGBoost saved successfully.
Saving report for model: CatBoost
Report for model CatBoost saved successfully.
All reports generated and saved to Summary_reports\Composite_score_20251011_005203.xlsx


In [19]:
all_results.keys()

dict_keys(['RandomForest', 'LightGBM', 'XGBoost', 'CatBoost'])

In [96]:
import joblib
# Saving the models and feature engineering parameters for future use
print("\n--- Saving Trained Models and Feature Engineering Parameters ---")
model_save_path = 'Trained_Models'
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

for model_name, results in all_results.items():
    artifact_toSave = {
        'trained_models': results['trained_models'],
        'feature_engineering_params': results['feature_engineering_params'],
    }

    run_prefix="Comp_score"

    if len(run_prefix)==0:
        file_path = f'{model_configs[model_name]["code"]}_artifacts.joblib'
    else:
        file_path = f'{run_prefix}_{model_configs[model_name]["code"]}_artifacts.joblib'

    # Create folder 'Model_Artifacts' if it doesn't exist
    if not os.path.exists('Model_Artifacts'):
        os.makedirs('Model_Artifacts')
    file_path = os.path.join('Model_Artifacts', file_path)

    # Check if a file already exists at that path
    if os.path.exists(file_path):
        print(f"Existing file found at '{file_path}'. Archiving it.")    
        # Create a timestamp string (e.g., "20250929_091303")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")    
        # Split the original file path into its base name and extension
        base_name, extension = os.path.splitext(file_path)    
        # Create the new name for the *old* file by inserting the timestamp
        archive_file_path = f"{base_name}_{timestamp}{extension}"    
        # Rename the old file
        os.rename(file_path, archive_file_path)
        print(f"Renamed existing file to '{archive_file_path}'")

    joblib.dump(artifact_toSave, file_path)

    print(f"Saved artifacts for model '{model_name}' to '{file_path}'")



--- Saving Trained Models and Feature Engineering Parameters ---
Existing file found at 'Model_Artifacts\Comp_score_RFR_artifacts.joblib'. Archiving it.
Renamed existing file to 'Model_Artifacts\Comp_score_RFR_artifacts_20251011_005207.joblib'
Saved artifacts for model 'RandomForest' to 'Model_Artifacts\Comp_score_RFR_artifacts.joblib'
Existing file found at 'Model_Artifacts\Comp_score_LGBM_artifacts.joblib'. Archiving it.
Renamed existing file to 'Model_Artifacts\Comp_score_LGBM_artifacts_20251011_005208.joblib'
Saved artifacts for model 'LightGBM' to 'Model_Artifacts\Comp_score_LGBM_artifacts.joblib'
Existing file found at 'Model_Artifacts\Comp_score_XGB_artifacts.joblib'. Archiving it.
Renamed existing file to 'Model_Artifacts\Comp_score_XGB_artifacts_20251011_005208.joblib'
Saved artifacts for model 'XGBoost' to 'Model_Artifacts\Comp_score_XGB_artifacts.joblib'
Existing file found at 'Model_Artifacts\Comp_score_CAT_artifacts.joblib'. Archiving it.
Renamed existing file to 'Model_A